In [0]:
employee=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/employee_project/employee_files/employee_duplicate_record.csv")

In [0]:
employee.show()

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|  EMP100001|   Devansh|  Thakkar|     Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|
|  EMP100002|   Yashoda|    Batra|     Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|
|  EMP100003|     Meera| Banerjee|     Sales|   Engineer| EMP100001| 598830|  2025-08-25| Bengaluru|         Active|
|  EMP100004|   Agastya|    Chana|     Sales|   Engineer| EMP100001|1174244|  2021-12-16| Bengaluru|         Active|
|  EMP100005|  Nachiket|     Bath|        HR|    Manager| EMP100001| -55000|  2025-02-14| Hyderabad|       Inactive|
|  EMP100006|      Avni|  Prakash|        IT|    Manager| EMP100

In [0]:
drop_duplicate=employee.dropDuplicates()
drop_duplicate.orderBy("employee_id").show(drop_duplicate.count())

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|  EMP100001|   Devansh|  Thakkar|     Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|
|  EMP100002|   Yashoda|    Batra|     Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|
|  EMP100003|     Meera| Banerjee|     Sales|   Engineer| EMP100001| 598830|  2025-08-25| Bengaluru|         Active|
|  EMP100004|   Agastya|    Chana|     Sales|   Engineer| EMP100001|1174244|  2021-12-16| Bengaluru|         Active|
|  EMP100005|  Nachiket|     Bath|        HR|    Manager| EMP100001| -55000|  2025-02-14| Hyderabad|       Inactive|
|  EMP100006|      Avni|  Prakash|        IT|    Manager| EMP100

In [0]:
print(f"Before:{employee.count()}")
print(f"After:{drop_duplicate.count()}")

Before:29
After:28


In [0]:
from pyspark.sql.functions import count,col
id=employee.groupBy("employee_id").agg(count("*").alias("count")).filter(col("count")>1)
id.show()

+-----------+-----+
|employee_id|count|
+-----------+-----+
|    EMP1004|    2|
|    EMP1003|    2|
+-----------+-----+



In [0]:
from pyspark.sql.window import Window
window_spe=Window.partitionBy("employee_id")

df_duplicate=employee.withColumn(
    "employee_id_count",
    count("employee_id").over(window_spe)
).filter(col("employee_id_count")>1)
df_duplicate.show(employee.count())

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+-----------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|     city|employee_status|employee_id_count|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+-----------------+
|    EMP1003|     Arjun| Banerjee|   Finance|   Engineer| EMP100001| 72000|  2025-08-25|Bengaluru|         Active|                2|
|    EMP1003|     Arjun| Banerjee|   Finance|   Engineer| EMP100001| 72000|  2025-08-25|Bengaluru|         Active|                2|
|    EMP1004|     Kavin|    Chana|        IT|   Engineer| EMP100001| 60000|  2021-12-16|Bengaluru|         Active|                2|
|    EMP1004|     Kavin|    Chana|        IT|   Engineer| EMP100001| 65000|  2021-12-16|Bengaluru|         Active|                2|
+-----------+----------+---------+----------+-----------+----------+-

In [0]:
remove_duplicate=employee.dropDuplicates()
remove_duplicate.show()

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|  EMP100020|       Avi|  Chander|        IT|    Analyst| EMP100001| -48000|  2024-05-24|      Pune|         Active|
|  EMP100021|     Amara|    Memon|     Sales|    Manager| EMP100001| 337778|  2024-07-18|    Mumbai|       Inactive|
|  EMP100010| Wriddhish|    Dugar|     Sales|   Engineer| EMP100001| 595442|  2026-04-23|    Mumbai|       Inactive|
|  EMP100012|      Owen|   Sankar|     Sales|    Manager| EMP100001| -72000|  2022-11-12|     Delhi|       Inactive|
|  EMP100007|Harinakshi|     Kota|        HR|    Manager| EMP100001| 408618|  2024-02-21|      Pune|       Inactive|
|  EMP100008|      Neha|  Dhillon|     Sales|   Engineer| EMP100

In [0]:
print(f"Original rows:{employee.count()}")
print(f"After removing exact duplicate:{remove_duplicate.count()}")

Original rows:29
After removing exact duplicate:28


In [0]:
df_duplicate=employee.dropDuplicates(["employee_id"]).orderBy("employee_id")
df_duplicate.show(df_duplicate.count())

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+
|  EMP100001|   Devansh|  Thakkar|     Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|
|  EMP100002|   Yashoda|    Batra|     Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|
|  EMP100003|     Meera| Banerjee|     Sales|   Engineer| EMP100001| 598830|  2025-08-25| Bengaluru|         Active|
|  EMP100004|   Agastya|    Chana|     Sales|   Engineer| EMP100001|1174244|  2021-12-16| Bengaluru|         Active|
|  EMP100005|  Nachiket|     Bath|        HR|    Manager| EMP100001| -55000|  2025-02-14| Hyderabad|       Inactive|
|  EMP100006|      Avni|  Prakash|        IT|    Manager| EMP100